# 21 Subsurface data

<!-- navigation -->
[<kbd>← 20 Geohydrological scheme</kbd>](20%20Geohydrological%20scheme.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>51 Prepare model inputs →</kbd>](51%20Prepare%20model%20inputs.ipynb)

**Optional**: support for notebook 20

Downloads subsurface data along each transect (BRO CPTs and boreholes, REGIS and GeoTOP) as support for the geohydrological scheme.

| | |
|---|---|
| **Input** | rectangle and cross-section line per transect in the transects metadata |
| **Output** | files and figures in the transect folder 'subsurface': BRO CPTs and boreholes (`bro`), REGIS and GeoTOP (`regis_geotop`), including `regis_geotop/regis_profile.xlsx` (REGIS layers in the middle of the cross-section) |

Downloads subsurface data that help to fill in the geohydrological scheme: the locations, graphs and logs of BRO cone penetration tests (CPT) and boreholes in the rectangle of each transect, and REGIS and GeoTOP along the cross-section line, including the REGIS layers in the middle of the cross-section as a table. Downloading can take several minutes per transect.

The data are downloaded with two packages, whose documentation describes the sources and the data in more detail:

- [brodata](https://github.com/ArtesiaWater/brodata): BRO data (CPTs, boreholes, monitoring wells) and DINO borehole samples;
- [nlmod](https://nlmod.readthedocs.io/): REGIS and GeoTOP (TNO), see the nlmod documentation on reading the layer models.

This notebook calls these packages directly, so it also shows how to use them; the toolkit (`tlt`) is only used for
the steps that are specific to this workflow (maps, saving files and figures, the REGIS table). Where a `tlt` function
wraps a package, its docstring names the call it uses (e.g. `help(tlt.download_cpt_graphs)`).

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import brodata
import matplotlib.pyplot as plt
import nlmod
import pandas as pd
tlt.show_versions()

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
## UPDATE FOR YOUR PROJECT
transects = project.transects()  # or a selection, e.g. ["Papendrecht West"]
download_bro = True              # CPTs and boreholes from the BRO
download_regis_geotop = True     # REGIS and GeoTOP along the cross-section line
zlim = None                      # (zmin, zmax) [m NAP] of the profiles, e.g. (-25.0, 3.0); None = derived from the BRO data

## 1. BRO data with brodata
For each data type, `get_characteristics(extent=...)` of [brodata](https://github.com/ArtesiaWater/brodata) returns a
table with one row per location in the rectangle of the transect (`xmin_extent` .. `ymax_extent` in the metadata),
indexed by the BRO id: cone penetration tests (`brodata.cpt`), geotechnical boreholes (`brodata.bhr`) and groundwater
monitoring wells (`brodata.gmw`). The DINO borehole samples come from `brodata.dino`. brodata warns about each XML tag
it does not read; `tlt.quiet_brodata()` hides these warnings (other warnings and errors are still shown).

The locations are saved in `subsurface/bro/bro_locations.gpkg` (one layer per data type), e.g. to view them in QGIS.
The vertical range of the profiles further on is `zlim`, or derived from the ground levels and depths of these
locations.

In [ ]:
meta_by_transect = project.metadata().set_index("transect")
bro_locations = {}  # per transect: the locations per data type
z_limits = {}       # per transect: the vertical range of the profiles [m NAP]
for transect in transects:
    m = meta_by_transect.loc[transect]
    extent = [m.get(c) for c in ("xmin_extent", "xmax_extent", "ymin_extent", "ymax_extent")]
    if download_bro and not any(pd.isna(v) for v in extent):
        with tlt.quiet_brodata():
            locations = {
                "cpt": brodata.cpt.get_characteristics(extent=extent),       # cone penetration tests
                "bhr": brodata.bhr.get_characteristics(extent=extent),       # geotechnical boreholes
                "gmw": brodata.gmw.get_characteristics(extent=extent),       # groundwater monitoring wells
                "gbo": brodata.dino.get_boormonsterprofiel(extent=extent),   # DINO borehole samples
            }
        # OLD style (until brodata PR #62 is released; toolkit issue #56): the DINO samples come without a
        # coordinate reference system (https://github.com/ArtesiaWater/brodata/issues/61); like the extent,
        # they are in RD New. NEW style: brodata sets EPSG:28992 itself; then remove these two lines.
        if locations["gbo"].crs is None:
            locations["gbo"] = locations["gbo"].set_crs(28992)
        print(f"{transect}: " + ", ".join(f"{len(gdf)} {kind}" for kind, gdf in locations.items()))
        tlt.save_bro_locations(locations, project.transect_dir(transect, "subsurface") / "bro" / "bro_locations.gpkg")
        bro_locations[transect] = locations

    if zlim is not None:
        z_limits[transect] = zlim
    elif transect in bro_locations:
        z_limits[transect] = tlt.suggest_z_limits(bro_locations[transect]) or (-30.0, 5.0)
    else:
        z_limits[transect] = (-30.0, 5.0)  # no BRO data to derive the range from
    print(f"{transect}: profiles from {z_limits[transect][0]:.0f} to {z_limits[transect][1]:.0f} m NAP")

In [ ]:
# The first locations of the first transect, as an example of the tables
for kind, gdf in next(iter(bro_locations.values()), {}).items():
    print(kind)
    display(gdf.head(3))

### 1.1 Map of the BRO locations

In [ ]:
for transect, locations in bro_locations.items():
    m = meta_by_transect.loc[transect]
    extent = [m[c] for c in ("xmin_extent", "xmax_extent", "ymin_extent", "ymax_extent")]
    f, ax = tlt.plot_bro_locations(locations, extent, line=project.crosssection(transect), title=transect)
    tlt.finalize_and_save(f, project.transect_dir(transect, "subsurface") / "bro", f"{project.name} - {transect}",
                          tag="bro_locations", file_label=transect)

### 1.2 CPTs and boreholes
For each CPT and borehole, `tlt.download_cpt_graphs` and `tlt.download_bhr_graphs` save the data (XML) and the graph or
log made by the BRO (SVG) in `subsurface/bro`. They use `brodata.cpt.ConePenetrationTest.from_bro_id(bro_id, to_file=...)`
and `brodata.cpt.graph(...)`, and `brodata.bhr.GeotechnicalBoreholeResearch.from_bro_id(bro_id, to_file=...)` and
`brodata.bhr.bhrgt_graph(...)`.

In [ ]:
for transect, locations in bro_locations.items():
    bro_folder = project.transect_dir(transect, "subsurface") / "bro"
    if len(locations["cpt"]) > 0:
        tlt.download_cpt_graphs(locations["cpt"], bro_folder)
    if len(locations["bhr"]) > 0:
        tlt.download_bhr_graphs(locations["bhr"], bro_folder, z_limits[transect])

The saved XML files can be read again with brodata, without downloading. As an example, the first CPT and the first
borehole of each transect: the measurements of a CPT are a table (`cpt.conePenetrationTest`), plotted with
`brodata.plot.cone_penetration_test`; the soil layers of a borehole are plotted with
`brodata.plot.descriptive_borehole_log`.

In [ ]:
for transect, locations in bro_locations.items():
    bro_folder = project.transect_dir(transect, "subsurface") / "bro"
    if len(locations["cpt"]) > 0:
        bro_id = locations["cpt"].index[0]
        with tlt.quiet_brodata():
            cpt = brodata.cpt.ConePenetrationTest(str(bro_folder / f"{bro_id}.xml"))
        print(f"{transect}: CPT {bro_id}")
        display(cpt.conePenetrationTest[["depth", "coneResistance", "localFriction", "frictionRatio"]].dropna().head())
        brodata.plot.cone_penetration_test(cpt, figsize=(6, 8))
    if len(locations["bhr"]) > 0:
        bro_id = locations["bhr"].index[0]
        with tlt.quiet_brodata():
            bhr = brodata.bhr.GeotechnicalBoreholeResearch(str(bro_folder / f"{bro_id}.xml"))
        f, ax = brodata.plot.descriptive_borehole_log(bhr)
        ax.set_title(f"{transect}: {bro_id}", loc="right")

### 1.3 DINO borehole samples along the cross-section
The lithology of the DINO borehole samples, projected on the cross-section line with
`brodata.plot.lithology_along_line`.

In [ ]:
for transect, locations in bro_locations.items():
    line = project.crosssection(transect)
    if line is None or len(locations["gbo"]) == 0:
        continue
    f, ax = plt.subplots(figsize=(10, 6))
    brodata.plot.lithology_along_line(locations["gbo"], line, ax=ax, kind="dino")
    ax.set_xlabel("distance along the cross-section [m]")
    ax.set_ylabel("z [m NAP]")
    ax.set_title(transect, loc="right")
    tlt.finalize_and_save(f, project.transect_dir(transect, "subsurface") / "bro", f"{project.name} - {transect}",
                          tag="dino_lithology", file_label=transect)

## 2. REGIS and GeoTOP with nlmod
[nlmod](https://nlmod.readthedocs.io/) downloads the layer models of TNO for a rectangle around the cross-section line
(the line plus 100 m, `tlt.extent_around_line`):

- REGIS with `nlmod.read.regis.download_regis`, here with the layer properties top, bottom, kh, kv, kD and c;
- GeoTOP (voxels of 100 x 100 x 0.5 m with the lithology class) with `nlmod.read.geotop.download_geotop`;
- the combined layer model: REGIS with the Holocene layer (HLc) replaced by the GeoTOP layers, with
  `tlt.combined_layer_model`. This is `nlmod.read.regis.get_combined_layer_models`, with a work-around for the layer
  names with pandas >= 3.

The results are xarray Datasets, see the nlmod documentation for more examples.

In [ ]:
layer_models = {}  # per transect: the Datasets 'regis', 'geotop' and 'combined'
for transect in transects:
    line = project.crosssection(transect)  # None if the metadata have no cross-section line
    if not download_regis_geotop or line is None:
        continue
    extent = tlt.extent_around_line(line)
    regis = nlmod.read.regis.download_regis(
        extent, variables=["top", "botm", "kh", "kv", "kD", "c"], drop_layer_dim_from_top=False
    )
    geotop = nlmod.read.geotop.download_geotop(extent, probabilities=True)
    combined = tlt.combined_layer_model(extent, geotop)
    layer_models[transect] = {"regis": regis, "geotop": geotop, "combined": combined}
    print(f"{transect}: {regis.sizes['layer']} REGIS layers, {combined.sizes['layer']} layers in the combined model")
# REGIS of the last transect, as an example of an nlmod Dataset
list(layer_models.values())[-1]["regis"] if layer_models else None

### 2.1 Layer models along the cross-section
REGIS and the combined model along the cross-section line, with `nlmod.plot.DatasetCrossSection` in the colours of the
REGIS legend.

In [ ]:
colors = nlmod.read.regis.get_legend(kind="combined")["color"].to_dict()
for transect, data in layer_models.items():
    line = project.crosssection(transect)
    zmin, zmax = z_limits[transect]
    f, axes = plt.subplots(2, 1, figsize=(14, 12), sharex=True)
    for ax, ds, title in zip(axes, (data["regis"], data["combined"]), ("REGIS", "REGIS + GeoTOP")):
        cs = nlmod.plot.DatasetCrossSection(ds, line=line, ax=ax, zmin=zmin, zmax=zmax)
        cs.plot_layers(colors=colors, min_label_area=1000)
        cs.plot_grid(linewidth=0.5, vertical=False)
        ax.set_ylabel("z [m NAP]")
        ax.set_title(f"{transect}: {title}", loc="right")
    axes[-1].set_xlabel("distance along the cross-section [m]")
    tlt.finalize_and_save(f, project.transect_dir(transect, "subsurface") / "regis_geotop",
                          f"{project.name} - {transect}", tag="layer_models", file_label=transect)

### 2.2 GeoTOP lithology
The lithology class of the GeoTOP voxels along the cross-section line, with `nlmod.plot.geotop_lithok_in_cross_section`.

In [ ]:
for transect, data in layer_models.items():
    line = project.crosssection(transect)
    zmin, zmax = z_limits[transect]
    f, ax = plt.subplots(figsize=(10, 5))
    nlmod.plot.geotop_lithok_in_cross_section(line, data["geotop"], ax=ax)
    ax.set_ylim(zmin, zmax)
    ax.set_xlabel("distance along the cross-section [m]")
    ax.set_ylabel("z [m NAP]")
    ax.set_title(f"{transect}: GeoTOP lithology", loc="right")
    tlt.finalize_and_save(f, project.transect_dir(transect, "subsurface") / "regis_geotop",
                          f"{project.name} - {transect}", tag="geotop_lithology", file_label=transect)

### 2.3 Hydraulic conductivity from GeoTOP
GeoTOP gives the lithology class of each voxel, not its hydraulic conductivity. nlmod assigns a kh to each class:
`nlmod.read.geotop.get_lithok_props()` gives its default values (the table below). `nlmod.read.geotop.add_kh_and_kv`
adds kh and kv to the voxels. The table has no kv, so kv = kh / `anisotropy`; with the default `anisotropy = 1` the kv
figure is the same as the kh figure. Set a larger anisotropy, or pass your own table with the columns `lithok`, `kh`
and `kv`, to use other values. nlmod also has a table per stratigraphic unit and lithology class
(`nlmod.read.geotop.get_kh_kv_table`), but only for Noord-Brabant and Noord- and Midden-Limburg.

These values are only a first indication for the geohydrological scheme: they are defaults per lithology class, not
measured at the transect. The figures are made with `tlt.plot_geotop_conductivity`.

In [ ]:
## UPDATE FOR YOUR PROJECT
anisotropy = 1.0  # kh / kv for the lithology classes (the table has no kv)

lithok_props = nlmod.read.geotop.get_lithok_props()  # default kh [m/d] per lithology class
lithok_props[["code", "name", "kh"]]

In [ ]:
for transect, data in layer_models.items():
    line = project.crosssection(transect)
    zmin, zmax = z_limits[transect]
    geotop_k = nlmod.read.geotop.add_kh_and_kv(data["geotop"], lithok_props, anisotropy=anisotropy)
    for variable in ("kh", "kv"):
        f, ax = tlt.plot_geotop_conductivity(geotop_k, line, variable=variable, zmin=zmin, zmax=zmax)
        ax.set_title(transect, loc="right")
        tlt.finalize_and_save(f, project.transect_dir(transect, "subsurface") / "regis_geotop",
                              f"{project.name} - {transect}", tag=f"geotop_{variable}", file_label=transect)

## 3. REGIS layers in the middle of each cross-section
The REGIS layers at the nearest REGIS cell to the middle of the cross-section line: one table for all transects. Empty cells: the value is not defined for the layer (e.g. no kh in an aquitard, no c in an aquifer). Per transect the table is saved as `regis_profile.xlsx` in the folder `subsurface/regis_geotop` of the transect.

In [ ]:
regis_profiles = {}
for transect, data in layer_models.items():
    # The REGIS cell nearest to the middle of the cross-section line
    line = project.crosssection(transect)
    x_mid, y_mid = (line[0][0] + line[1][0]) / 2, (line[0][1] + line[1][1]) / 2
    regis_profiles[transect] = tlt.regis_profile(data["regis"], x_mid, y_mid)
    tlt.export_to_excel(regis_profiles[transect], project.transect_dir(transect, "subsurface") / "regis_geotop" / "regis_profile.xlsx")

if regis_profiles:
    profiles = pd.concat(regis_profiles, names=["transect"])
    display(profiles.style.format(precision=2, na_rep="-"))
else:
    print("No REGIS profiles (no cross-section line, or download_regis_geotop = False)")

---

<!-- navigation -->
[<kbd>← 20 Geohydrological scheme</kbd>](20%20Geohydrological%20scheme.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>51 Prepare model inputs →</kbd>](51%20Prepare%20model%20inputs.ipynb)